# 9.4 Publication Text Analysis - step 4: Topic Modeling

This notebook:

1. Fits a topic model (NMF, with LDA as a comparison) on the publication-level text corpus
2. Picks a number of topics k and labels each topic from its top words
3. Checks whether topics are broad, cross-lab patterns or artifacts of one or two prolific labs (the same issue we found with individual terms in 9_3)
4. Aggregates to lab-level topic shares for the regression in 9_5
5. Preliminary logistic regression with Lasso to see how well topics predict energy use

In [1]:
# Set up
import pandas as pd
import numpy as np
import sys
from pathlib import Path
CODE_ROOT = Path.cwd().parents[1]
sys.path.append(str(CODE_ROOT))
import config
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF, LatentDirichletAllocation
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.model_selection import cross_val_score, StratifiedKFold

In [2]:
# Load data
publications = pd.read_csv(
    config.PUBLICATON_DATA / "3_Clean" / "publications_unique.csv"
)

pub_lab_map = pd.read_csv(
    config.PUBLICATON_DATA / "3_Clean" / "pub_lab_mapping.csv"
)

lab_data = pd.read_csv(
    config.PUBLICATON_DATA / "3_Clean" / "lab_level_subject_shares.csv"
)

print(f"Publications: {publications.shape[0]:,}")
print(f"Pub-lab mapping rows: {pub_lab_map.shape[0]:,}")
print(f"Labs: {lab_data.shape[0]}")

Publications: 8,014
Pub-lab mapping rows: 8,435
Labs: 95


## (1) Create TF-IDF matrix

We use same parameters as in 9_3 (min_df = 3, max_df = 0.95, unigrams and bigrams, English stopwords).

In [3]:
# Create TF-IDF matrix
tfidf = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2),
    max_df=0.95,
    min_df=3,
)
X = tfidf.fit_transform(publications["text_final"])
vocab = tfidf.get_feature_names_out()

print(f"Matrix shape: {X.shape[0]:,} documents x {X.shape[1]:,} vocabulary terms")

Matrix shape: 8,014 documents x 44,351 vocabulary terms


## (2) Topic modelling - first attempt (choose k = 15)

In [4]:
# Choose k = 15 topics for the first attempt
N_TOPICS_FIRST_PASS = 15

# Fit NMF model
nmf = NMF(n_components=N_TOPICS_FIRST_PASS, random_state=0, max_iter=500)
doc_topics = nmf.fit_transform(X)  # documents x topics
topic_terms = nmf.components_       # topics x terms

print(f"doc_topics shape: {doc_topics.shape}")
print(f"topic_terms shape: {topic_terms.shape}")

doc_topics shape: (8014, 15)
topic_terms shape: (15, 44351)


In [5]:
# Display top 10 words for each topic
N_TOP_WORDS = 10

for topic_idx, topic in enumerate(topic_terms):
    top_word_idx = topic.argsort()[::-1][:N_TOP_WORDS]
    top_words = [vocab[i] for i in top_word_idx]
    print(f"Topic {topic_idx}: {', '.join(top_words)}")

Topic 0: proton, 13 tev, sqrt 13, 13, proton proton, proton collisions, tev, sqrt, collisions sqrt, collisions
Topic 1: value documentation, documentation, mak value, value, mak, documentation german, german language, language, 2018, language 2018
Topic 2: species, diversity, plant, soil, richness, species richness, biodiversity, forest, tree, climate
Topic 3: pi, pm, decays, cp, pi pi, decay, psi, violation, cp violation, rightarrow
Topic 4: mak, value, m3, mak value, mg, toxicity, ml m3, ml, commission, rats
Topic 5: prodromus, prodromus fern, flora bolivia, fern flora, bolivia, flora, fern, new, dryopteridaceae, sm
Topic 6: ebv, cells, cell, immune, virus, human, infection, nk, mice, barr
Topic 7: qcd, boson, production, corrections, higgs, order, nnlo, jet, higgs boson, leading
Topic 8: german version, translation german, translation, version, supplement, supplement translation, mak, documentation supplement, value documentation, documentation
Topic 9: neuromorphic, neural, circuit

We see that the topic categories seem to make sense - we see similar patterns to what we saw for the top words in 9_3 - biology/wet labs (topic 6), mak docs (topic 4). We also see several topics regarding particle physics type research (topics 0, 3, 6, 10, 13).

## (3) Check whether each topic is broad pattern or reflecting 1/2 labs

In [6]:
# Each publication's single dominant topic (highest weight)
publications["dominant_topic"] = doc_topics.argmax(axis=1)

# Join to labs via the pub-lab mapping
pub_topic_lab = pub_lab_map.merge(
    publications[["pub_id", "dominant_topic"]], on="pub_id", how="left"
)

# Summarize topics by number of pubs, number of distinct labs, and share of pubs from the top lab
def top_lab_share(group):
    counts = group["labgroupid"].value_counts()
    return counts.iloc[0] / counts.sum()

topic_summary = pd.DataFrame({
    "n_pubs": pub_topic_lab.groupby("dominant_topic").size(),
    "n_distinct_labs": pub_topic_lab.groupby("dominant_topic")["labgroupid"].nunique(),
    "top_lab_share": pub_topic_lab.groupby("dominant_topic").apply(top_lab_share),
})

def get_top_words(topic_idx, n=8):
    top_word_idx = topic_terms[topic_idx].argsort()[::-1][:n]
    return ", ".join(vocab[i] for i in top_word_idx)

topic_summary["top_words"] = [get_top_words(i) for i in topic_summary.index]

pd.set_option("display.max_colwidth", 100)
print(topic_summary.sort_values("n_distinct_labs").to_string())

                n_pubs  n_distinct_labs  top_lab_share                                                                                                                    top_words
dominant_topic                                                                                                                                                                     
5                   87                2       0.942529                                              prodromus, prodromus fern, flora bolivia, fern flora, bolivia, flora, fern, new
8                  136                6       0.948529  german version, translation german, translation, version, supplement, supplement translation, mak, documentation supplement
10                 158                6       0.506329                                                    mu, mu mu, mu decays, decay, rightarrow, rightarrow mu, decays, branching
1                  190                8       0.926316                   value documentation, docume

/var/folders/nn/21zflm3n7gzc5spw42wq352rpm87xt/T/ipykernel_68786/2565383140.py:17: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  "top_lab_share": pub_topic_lab.groupby("dominant_topic").apply(top_lab_share),


## (4) Aggregate to lab-level topic shares

In [7]:
# Normalize each publication's topic weights to a proper distribution (sums to 1)
zero_weight_docs = (doc_topics.sum(axis=1) == 0).sum()
print(f"Publications with no usable vocab (excluded from topic shares): {zero_weight_docs}")

with np.errstate(invalid="ignore"):
    doc_topic_shares = doc_topics / doc_topics.sum(axis=1, keepdims=True)

topic_cols = [f"topic_{i}" for i in range(doc_topic_shares.shape[1])]
doc_topic_df = pd.DataFrame(doc_topic_shares, columns=topic_cols)
doc_topic_df["pub_id"] = publications["pub_id"].values

# Join to labs, then average within each lab across that lab's own publications
pub_lab_topics = pub_lab_map.merge(doc_topic_df, on="pub_id", how="left")
lab_topic_shares = pub_lab_topics.groupby("labgroupid")[topic_cols].mean().reset_index()

print(f"lab_topic_shares shape: {lab_topic_shares.shape}")
print(f"Each lab's topic shares still sum to ~1: {lab_topic_shares[topic_cols].sum(axis=1).describe()[['min','max']].to_dict()}")
# lab_topic_shares.head(3)

Publications with no usable vocab (excluded from topic shares): 36
lab_topic_shares shape: (95, 16)
Each lab's topic shares still sum to ~1: {'min': 0.9999999999999998, 'max': 1.0000000000000002}


## (5) Quick check: do topic shares predict high/low energy use?

We do the real analysis in 9_5 but this is just a quick check to see whether the topic shares are predictive.

In [8]:
# Create high/low energy tier variable
median_energy = lab_data["annual_electricity_total"].median()
lab_data["energy_tier"] = np.where(lab_data["annual_electricity_total"] >= median_energy, 1, 0)

# Merge the topic shares data with the energy tier data
model_data = lab_topic_shares.merge(lab_data[["labgroupid", "energy_tier"]], on="labgroupid", how="inner")
print(f"Labs with both topic shares and an energy tier: {len(model_data)}")

# Prepare the data for modelling
X_topics = model_data[topic_cols].values
y = model_data["energy_tier"].values

Labs with both topic shares and an energy tier: 95


In [9]:
# Nested cross-validation to evaluate Lasso logistic regression with hyperparameter tuning (C)
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

# We use LogisticRegressionCV to perform inner cross-validation for hyperparameter tuning (C)
# - penalty="l1" = Lasso
# - Cs=10 searches 10 values on a log scale
# - cv=5 is the inner cross-validation used to pick C
lasso_clf = LogisticRegressionCV(
    Cs=10,
    cv=5,
    penalty="l1",
    solver="liblinear",
    max_iter=1000,
    random_state=0,
)

# Perform nested cross-validation to evaluate the model's performance
cv_accuracy_lasso = cross_val_score(lasso_clf, X_topics, y, cv=outer_cv, scoring="accuracy")
cv_auc_lasso = cross_val_score(lasso_clf, X_topics, y, cv=outer_cv, scoring="roc_auc")

# Report results
print(f"Nested CV accuracy (Lasso, tuned C): {cv_accuracy_lasso.mean():.1%} (+/- {cv_accuracy_lasso.std():.1%} across folds)")
print(f"Nested CV ROC-AUC (Lasso, tuned C):  {cv_auc_lasso.mean():.3f} (+/- {cv_auc_lasso.std():.3f} across folds)")

Nested CV accuracy (Lasso, tuned C): 81.1% (+/- 5.4% across folds)
Nested CV ROC-AUC (Lasso, tuned C):  0.842 (+/- 0.093 across folds)


We now investigate which topics are actually kept by Lasso.

In [10]:
# Fit the final model on the full dataset to inspect coefficients
lasso_full = LogisticRegressionCV(
    Cs=10, 
    cv=5, 
    penalty="l1", 
    solver="liblinear", 
    max_iter=1000, 
    random_state=0,
)
lasso_full.fit(X_topics, y)

# Create df to display results
coef_df = pd.DataFrame({
    "topic": topic_cols,
    "coefficient": lasso_full.coef_[0],
    "top_words": [get_top_words(i, n=6) for i in range(len(topic_cols))],
})
coef_df["abs_coef"] = coef_df["coefficient"].abs()

# Report results
n_nonzero = (coef_df["coefficient"] != 0).sum()
print(f"Selected C: {lasso_full.C_[0]:.4f}")
print(f"Topics with non-zero coefficient: {n_nonzero} / {len(topic_cols)}")
print()
print(coef_df.sort_values("abs_coef", ascending=False).drop(columns="abs_coef").to_string(index=False))

Selected C: 2.7826
Topics with non-zero coefficient: 5 / 15

   topic  coefficient                                                                                    top_words
 topic_6     8.543398                                                       ebv, cells, cell, immune, virus, human
topic_14     1.842460                                                  ms, determination, air, urine, method, acid
topic_12    -1.071304                                                    cancer, risk, mortality, 95, patients, ci
 topic_2    -0.621676                                  species, diversity, plant, soil, richness, species richness
 topic_9    -0.100775                                   neuromorphic, neural, circuits, learning, spiking, network
 topic_0     0.000000                                proton, 13 tev, sqrt 13, 13, proton proton, proton collisions
 topic_1     0.000000              value documentation, documentation, mak value, value, mak, documentation german
 topic_3     0.0000